# Method: requirements, gates, and a metric that rewarded mediocrity

*Question → Intuition → Math → Code → Assumptions → How it breaks*

## 1. Question

Given a list of things greatness requires, how do you combine them into one
ranking without letting a player be terrible at one of them and win anyway?

## 2. Intuition

Two obvious options, and they behave very differently.

**Average them.** Simple, robust, familiar. But it permits compensation: score
enough goals and it no longer matters that you are never available.

**Gate on them.** Require a minimum on every requirement, then rank whoever
clears the bar. This takes "must have" literally.

The project gates, because the word "must" was in the definition.

## 3. Math

For player $i$ and requirement $k$, with standardised score $z_{ik}$ and floor
$f_k$ set at percentile $p$ of the population:

$$\text{qualified}_i = \bigwedge_k \left( z_{ik} \ge f_k \right)$$

$$\text{score}_i = \frac{\sum_k w_k z_{ik}}{\sum_k w_k}$$

The gate is a logical AND, so it is **unforgiving by design**: eleven
requirements at the 40th percentile would admit only $0.6^{11} \approx 0.36\%$
of players if the requirements were independent. They are correlated, so the
real figure is higher — but the gate still does the bulk of the work, and the
weights $w_k$ only order the survivors.

In [ ]:
import warnings

import matplotlib
import numpy as np
import pandas as pd

from gambeta import needs

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")
keepers = pd.read_parquet(f"{SAMPLE}/keeper_ranking.parquet")

qualified = ranking[ranking["qualified"]]
independent = 0.6**11
print(f"if requirements were independent: {100 * independent:.2f}% would qualify")
print(f"actually qualified:                {100 * len(qualified) / len(ranking):.2f}%")
print("\nrequirements are strongly correlated - being good at one predicts the others")

## 4. Code

Raising the floor tightens the gate. This is the single most consequential knob
in the project and it is a judgement call, so it is exposed rather than
buried.

In [ ]:
import dataclasses

from gambeta import gate, kit

cfg = kit.load()
keys = [r.key for r in needs.OUTFIELD if r.key in ranking.columns]
profile = ranking[["player_id", "player", "seasons", "leagues", *keys]].copy()

for pct in (20, 30, 40, 50, 60):
    out = gate.qualify_and_rank(
        profile, needs.OUTFIELD, dataclasses.replace(cfg, gate_percentile=pct)
    )
    print(f"  floor at {pct}th percentile -> {int(out['qualified'].sum()):4d} qualify")

## 5. Assumptions

1. **Every requirement is genuinely necessary.** The gate treats them as
   mandatory, so including a bad requirement does more damage here than in an
   average, where it would merely be diluted.
2. **The floor is meaningful at the same percentile for all of them.** There is
   no reason the 40th percentile of discipline is as demanding as the 40th
   percentile of scoring.
3. **Standardising across players makes them comparable.** Save percentage and
   goals per 90 are only on one scale because we forced them onto one.

## 6. How it breaks

It broke, on the first real run, in a way worth showing in full.

`consistency` was defined as **−(standard deviation of a player's season
scores)** — the intuition being that a metronome is better than a streaky
player. Watch what that does.

In [ ]:
elite = np.array([2.5, 4.0, 3.0, 4.0, 2.5, 3.5])  # a great player's seasons
flat = np.array([-0.1, 0.1, -0.1, 0.1, -0.1, 0.1])  # a journeyman's

print(f"elite      mean {elite.mean():+.2f}   sd {elite.std():.2f}   old score {-elite.std():+.2f}")
print(f"journeyman mean {flat.mean():+.2f}   sd {flat.std():.2f}   old score {-flat.std():+.2f}")
print("\nThe journeyman scores far better on 'consistency'.")

**Variance is anti-correlated with excellence.** An elite player swings between
very good and outstanding, so their standard deviation is large. A journeyman
sits flat at mediocre, so theirs is near zero.

Inside a gate, that is fatal. On the first run this single requirement
disqualified Messi, Ronaldo, Kane, Haaland, Lewandowski, Suárez, Henry and Salah
simultaneously — and the top qualifier was a player nobody would nominate.

The fix was to change what the requirement measures, not its threshold.
"Does he have bad years?" is a question about a **floor**, not a spread:

$$\text{consistency}_i = \text{percentile}_{20}\left(s_{i1}, \dots, s_{iT}\right)$$

A great player's twentieth-percentile season is still good.

In [ ]:
print(f"elite      20th pct {np.percentile(elite, 20):+.2f}")
print(f"journeyman 20th pct {np.percentile(flat, 20):+.2f}")
print("\nNow the great player wins, which is the point.")

### The general lesson

A metric can be perfectly correct as arithmetic and completely wrong as a
measurement. Nothing about `-std()` is a bug; it computes exactly what it says.
The error was believing that low variance means quality.

**This is why the sanity check exists.** No test caught it — every unit test
passed. It was caught by looking at the output and recognising that the answer
was absurd. Domain knowledge is a debugging tool, and on this project it was the
only one that would have worked.